# Taller 1 — Consumo Automatizado de APIs

**Asignatura:** MLY1101 — Machine Learning  
**Nombre del estudiante:** Gaete Francisco
**Sección:** MLY1101_001V  
**Fecha:** 26-09-2026

## Pregunta u objetivo

> **¿Qué información puede recopilarse sobre el anime desde tres perspectivas: series, personajes y estudios de animación?**

Cada API aporta una perspectiva distinta del mismo tema:

| # | API | Perspectiva | Archivo |
|---|-----|-------------|---------|
| 1 | Shikimori | Series de anime ordenadas por ranking | `dataset_api_1.csv` |
| 2 | Kitsu | Personajes de anime | `dataset_api_2.csv` |
| 3 | AniList (GraphQL) | Estudios de animación | `dataset_api_3.csv` |

Las tres APIs están listadas en la categoría **Anime** de https://github.com/public-apis/public-apis y no requieren API key.  
*No se responde la pregunta ni se integran las fuentes: cada dataset se guarda de forma independiente.*


## Consideraciones generales

- Se utilizan **3 APIs diferentes** disponibles en: https://github.com/public-apis/public-apis
- Cada API aporta información relacionada con el mismo objetivo.
- Se obtienen **mínimo 200 registros por API** mediante paginación.
- Cada API genera un archivo `.csv` independiente.
- **No se realiza merge, join, concat ni cruces entre datasets.**
- El notebook se ejecuta completo con **Entorno de ejecución → Ejecutar todas**.


In [1]:
# Librerías base
import time
import json
import requests
import pandas as pd
from pathlib import Path

# Los datasets se guardan en la carpeta actual (raíz del repositorio),
# tal como exige la estructura de entrega en GitHub.
OUTPUT_DIR = Path('.')
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS_BASE = {'User-Agent': 'MLY1101-Taller1/1.0 (proyecto academico)'}

print(f'Carpeta de salida: {OUTPUT_DIR.resolve()}')


Carpeta de salida: /content


In [2]:


def solicitar(method, url, max_intentos=5, **kwargs):
    kwargs.setdefault('timeout', 30)
    for intento in range(1, max_intentos + 1):
        try:
            r = requests.request(method, url, **kwargs)
        except requests.RequestException as e:
            espera = 2 ** intento
            print(f'  Error de conexión ({e}). Reintento {intento}/{max_intentos} en {espera}s')
            time.sleep(espera)
            continue

        if r.status_code == 429 or r.status_code >= 500:
            try:
                espera = int(r.headers.get('Retry-After', 2 ** intento))
            except ValueError:
                espera = 2 ** intento
            print(f'  Status {r.status_code}. Reintento {intento}/{max_intentos} en {espera}s')
            time.sleep(espera)
            continue

        r.raise_for_status()
        return r

    raise RuntimeError(f'No se pudo obtener {url} tras {max_intentos} intentos')


def unir_nombres(lista, clave='name'):
    """Convierte una lista de diccionarios en un texto separado por comas."""
    return ', '.join(str(x.get(clave, '')) for x in (lista or []) if isinstance(x, dict))


# Fuente 1 — Shikimori

**Nombre de la API:** Shikimori API v1 (REST)  
**Documentación:** https://shikimori.io/api/doc  
**Endpoint utilizado:** `GET https://shikimori.io/api/animes?order=ranked&limit=50&page={n}`  
**Descripción de los datos:** Catálogo de series de anime ordenadas por ranking: nombre, tipo, puntuación, estado, episodios, fechas de emisión y URL.  
**Relación con el objetivo:** Aporta la perspectiva de las **series** de anime.

**Paginación:** hasta 50 registros por página; se recorre `page = 1, 2, 3...` hasta reunir 200 registros o hasta recibir una página vacía.  
**Límite de uso:** 5 solicitudes/segundo y 90/minuto; exige un `User-Agent` identificable. Los endpoints de lectura no requieren autenticación.

> **Nota:** originalmente se planificó usar Jikan (MyAnimeList), pero en septiembre de 2026 esa API respondía HTTP 504 en todos sus endpoints por una caída del servicio, por lo que se reemplazó por Shikimori, otra API de anime del mismo repositorio.


In [3]:
# CONFIGURACIÓN API 1
API1_DOMINIOS = ['https://shikimori.io', 'https://shikimori.one']
API1_RUTA = '/api/animes'
API1_MIN_REGISTROS = 200

api1_headers = dict(HEADERS_BASE)   # Shikimori exige User-Agent
api1_params = {
    'order': 'ranked',   # ordenadas por ranking
    'limit': 50,         # máximo permitido por página
    'censored': 'true',  # excluye contenido adulto
}
API1_PAUSA = 1.0

# Seleccionar el primer dominio que responda correctamente
API1_URL = None
for dominio in API1_DOMINIOS:
    try:
        r = requests.get(dominio + API1_RUTA, headers=api1_headers,
                         params={**api1_params, 'limit': 1, 'page': 1}, timeout=30)
        if r.ok:
            API1_URL = dominio + API1_RUTA
            break
        print(f'{dominio}: status {r.status_code}')
    except requests.RequestException as e:
        print(f'{dominio}: sin respuesta ({e})')

if API1_URL is None:
    raise RuntimeError('Ningún dominio de Shikimori respondió.')
print('Endpoint API 1:', API1_URL)


Endpoint API 1: https://shikimori.io/api/animes


In [4]:
# CONSUMO API 1 (con paginación)
api1_raw = {}   # diccionario por id para evitar duplicados entre páginas
page = 1

while len(api1_raw) < API1_MIN_REGISTROS:
    params = {**api1_params, 'page': page}
    response = solicitar('GET', API1_URL, headers=api1_headers, params=params)
    nuevos = response.json()   # Shikimori devuelve directamente una lista

    print(f'Status API 1: {response.status_code} | página {page} | {len(nuevos)} registros')
    if not nuevos:
        print('No hay más páginas disponibles.')
        break

    for anime in nuevos:
        api1_raw[anime['id']] = anime

    page += 1
    time.sleep(API1_PAUSA)

# Selección y aplanado de campos relevantes
base_sitio = API1_URL.replace(API1_RUTA, '')
api1_registros = []
for a in api1_raw.values():
    url_rel = a.get('url') or ''
    api1_registros.append({
        'shikimori_id': a.get('id'),
        'titulo': a.get('name'),
        'titulo_ruso': a.get('russian'),
        'tipo': a.get('kind'),
        'puntuacion': a.get('score'),
        'estado': a.get('status'),
        'episodios': a.get('episodes'),
        'episodios_emitidos': a.get('episodes_aired'),
        'fecha_inicio': a.get('aired_on'),
        'fecha_fin': a.get('released_on'),
        'url': base_sitio + url_rel if url_rel.startswith('/') else url_rel,
    })

print('Registros API 1:', len(api1_registros))


Status API 1: 200 | página 1 | 50 registros
Status API 1: 200 | página 2 | 50 registros
Status API 1: 200 | página 3 | 50 registros
Status API 1: 200 | página 4 | 50 registros
Registros API 1: 200


In [5]:
# GUARDAR DATASET API 1
API1_ARCHIVO = OUTPUT_DIR / 'dataset_api_1.csv'

df_api1 = pd.DataFrame(api1_registros)
df_api1.to_csv(API1_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API1_ARCHIVO)
print('Registros guardados:', len(df_api1))
df_api1.head()


Archivo generado: dataset_api_1.csv
Registros guardados: 200


,shikimori_id,titulo,titulo_ruso,tipo,puntuacion,estado,episodios,episodios_emitidos,fecha_inicio,fecha_fin,url
0,52991,Sousou no Frieren,Провожающая в последний путь Фрирен,tv,9.25,released,28,27,2023-09-29,2024-03-22,https://shikimori.io/animes/52991-sousou-no-fr...
1,5114,Fullmetal Alchemist: Brotherhood,Стальной алхимик: Братство,tv,9.11,released,64,0,2009-04-05,2010-07-04,https://shikimori.io/animes/z5114-fullmetal-al...
2,61316,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,Re:Zero. Жизнь с нуля в альтернативном мире 4,tv,9.11,ongoing,19,18,2026-04-08,2026-09-30,https://shikimori.io/animes/61316-re-zero-kara...
3,61469,Steel Ball Run: JoJo no Kimyou na Bouken,Невероятное приключение ДжоДжо: Гонка «Стально...,ona,9.07,ongoing,0,2,2026-03-19,None,https://shikimori.io/animes/61469-steel-ball-r...
4,9253,Steins;Gate,Врата Штейна,tv,9.07,released,24,24,2011-04-06,2011-09-14,https://shikimori.io/animes/z9253-steins-gate


# Fuente 2 — Kitsu

**Nombre de la API:** Kitsu API (JSON:API)  
**Documentación:** https://kitsu.docs.apiary.io/  
**Endpoint utilizado:** `GET https://kitsu.app/api/edge/characters?page[limit]=20&page[offset]={n}`  
**Descripción de los datos:** Personajes de anime: nombre canónico, nombre en japonés, nombres alternativos, identificador en MyAnimeList, descripción e imagen.  
**Relación con el objetivo:** Aporta la perspectiva de los **personajes** de anime.

**Paginación:** máximo 20 registros por página. Se sigue el enlace `links.next` que entrega la propia API hasta reunir 200 registros o hasta que no exista página siguiente.


In [6]:
# CONFIGURACIÓN API 2
API2_URL = 'https://kitsu.app/api/edge/characters'
API2_MIN_REGISTROS = 200

api2_headers = {
    **HEADERS_BASE,
    'Accept': 'application/vnd.api+json',
    'Content-Type': 'application/vnd.api+json',
}
api2_params = {'page[limit]': 20, 'page[offset]': 0}   # 20 = máximo permitido
API2_PAUSA = 0.5


In [7]:
# CONSUMO API 2 (paginación siguiendo links.next)
api2_raw = {}
url = API2_URL
params = dict(api2_params)
pagina = 1

while url and len(api2_raw) < API2_MIN_REGISTROS:
    response = solicitar('GET', url, headers=api2_headers, params=params)
    data = response.json()

    nuevos = data.get('data', [])
    print(f'Status API 2: {response.status_code} | página {pagina} | {len(nuevos)} registros')
    if not nuevos:
        break

    for personaje in nuevos:
        api2_raw[personaje['id']] = personaje

    # El enlace 'next' ya incluye los parámetros de paginación
    url = data.get('links', {}).get('next')
    params = None
    pagina += 1
    time.sleep(API2_PAUSA)

# Selección y aplanado de campos relevantes
api2_registros = []
for p in api2_raw.values():
    attr = p.get('attributes', {}) or {}
    nombres = attr.get('names') or {}
    imagen = attr.get('image') or {}
    api2_registros.append({
        'kitsu_id': p.get('id'),
        'slug': attr.get('slug'),
        'nombre': attr.get('canonicalName') or attr.get('name'),
        'nombre_ingles': nombres.get('en'),
        'nombre_japones': nombres.get('ja_jp'),
        'otros_nombres': ', '.join(attr.get('otherNames') or []),
        'mal_id': attr.get('malId'),
        'descripcion': attr.get('description'),
        'imagen': imagen.get('original'),
        'creado': attr.get('createdAt'),
        'actualizado': attr.get('updatedAt'),
    })

print('Registros API 2:', len(api2_registros))


Status API 2: 200 | página 1 | 20 registros
Status API 2: 200 | página 2 | 20 registros
Status API 2: 200 | página 3 | 20 registros
Status API 2: 200 | página 4 | 20 registros
Status API 2: 200 | página 5 | 20 registros
Status API 2: 200 | página 6 | 20 registros
Status API 2: 200 | página 7 | 20 registros
Status API 2: 200 | página 8 | 20 registros
Status API 2: 200 | página 9 | 20 registros
Status API 2: 200 | página 10 | 20 registros
Registros API 2: 200


In [8]:
# GUARDAR DATASET API 2
API2_ARCHIVO = OUTPUT_DIR / 'dataset_api_2.csv'

df_api2 = pd.DataFrame(api2_registros)
df_api2.to_csv(API2_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API2_ARCHIVO)
print('Registros guardados:', len(df_api2))
df_api2.head()


Archivo generado: dataset_api_2.csv
Registros guardados: 200


,kitsu_id,slug,nombre,nombre_ingles,nombre_japones,otros_nombres,mal_id,descripcion,imagen,creado,actualizado
0,1,jet-black,Jet Black,Jet Black,ジェット・ブラック,"Running Rock, Black Dog",3,"Jet, known on his home satellite as the ""Black...",https://media.kitsu.app/characters/images/1/or...,2013-02-20T18:49:20.901Z,2021-09-25T06:59:59.531Z
1,2,spike-spiegel,Spike Spiegel,Spike Spiegel,スパイク・スピーゲル,,1,"Height: 185 cm (6' 1"") Weight: 70 kg (155 lbs)...",https://media.kitsu.app/characters/images/2/or...,2013-02-20T18:49:20.942Z,2021-09-25T06:59:59.942Z
2,3,faye-valentine,Faye Valentine,Faye Valentine,フェイ・バレンタイン,,2,"Faye Valentine Birthday: August 14, 1994 One ...",https://media.kitsu.app/characters/images/3/or...,2013-02-20T18:49:20.997Z,2021-09-25T06:59:59.164Z
3,4,edward-wong-hau-pepelu-tivrusky-iv,Edward Wong Hau Pepelu Tivrusky IV,Edward Wong Hau Pepelu Tivrusky IV,エドワード・ウォン・ハウ・ペペル・チブルスキー4世,"Ed, Radical Ed, FranÃ§oise",16,The self-invented personal name of an elite ne...,https://media.kitsu.app/characters/images/4/or...,2013-02-20T18:49:21.097Z,2021-09-25T06:59:59.807Z
4,5,ein-654ddb2c-eb4e-43b8-b7da-21b2b9ae8f68,Ein,Ein,アイン,,4,Ein is a Pembroke Welsh Corgi brought aboard t...,https://media.kitsu.app/characters/images/5/or...,2013-02-20T18:49:21.142Z,2021-09-25T06:59:59.458Z


# Fuente 3 — AniList

**Nombre de la API:** AniList API v2 (GraphQL)  
**Documentación:** https://docs.anilist.co/  
**Endpoint utilizado:** `POST https://graphql.anilist.co` (consulta `Page { studios }`)  
**Descripción de los datos:** Estudios de animación ordenados por cantidad de favoritos: nombre, si es estudio de animación, número de favoritos, URL y sus 3 obras más populares.  
**Relación con el objetivo:** Aporta la perspectiva de los **estudios** que producen anime.

**Paginación:** GraphQL con variables `page` y `perPage` (máximo 50). Se avanza de página hasta reunir 200 registros o hasta que `pageInfo.hasNextPage` sea `false`.


In [9]:
# CONFIGURACIÓN API 3
API3_URL = 'https://graphql.anilist.co'
API3_MIN_REGISTROS = 200

api3_headers = {
    **HEADERS_BASE,
    'Content-Type': 'application/json',
    'Accept': 'application/json',
}
API3_POR_PAGINA = 50   # máximo permitido por AniList
API3_PAUSA = 1.0

API3_QUERY = '''
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    pageInfo {
      currentPage
      hasNextPage
    }
    studios(sort: FAVOURITES_DESC) {
      id
      name
      isAnimationStudio
      favourites
      siteUrl
      media(sort: POPULARITY_DESC, perPage: 3) {
        nodes {
          title { romaji }
        }
      }
    }
  }
}
'''


In [10]:
# CONSUMO API 3 (paginación GraphQL)
api3_raw = {}
page = 1
hay_mas = True

while hay_mas and len(api3_raw) < API3_MIN_REGISTROS:
    payload = {'query': API3_QUERY, 'variables': {'page': page, 'perPage': API3_POR_PAGINA}}
    response = solicitar('POST', API3_URL, headers=api3_headers, json=payload)
    data = response.json()

    if 'errors' in data:
        raise RuntimeError(f'Error GraphQL: {data["errors"]}')

    pagina = data['data']['Page']
    nuevos = pagina.get('studios', [])
    print(f'Status API 3: {response.status_code} | página {page} | {len(nuevos)} registros')
    if not nuevos:
        break

    for estudio in nuevos:
        api3_raw[estudio['id']] = estudio

    hay_mas = pagina.get('pageInfo', {}).get('hasNextPage', False)
    page += 1
    time.sleep(API3_PAUSA)

# Selección y aplanado de campos relevantes
api3_registros = []
for s in api3_raw.values():
    obras = ((s.get('media') or {}).get('nodes')) or []
    api3_registros.append({
        'anilist_id': s.get('id'),
        'nombre': s.get('name'),
        'es_estudio_animacion': s.get('isAnimationStudio'),
        'favoritos': s.get('favourites'),
        'obras_mas_populares': ', '.join((o.get('title') or {}).get('romaji') or '' for o in obras),
        'url': s.get('siteUrl'),
    })

print('Registros API 3:', len(api3_registros))


Status API 3: 200 | página 1 | 50 registros
Status API 3: 200 | página 2 | 50 registros
Status API 3: 200 | página 3 | 50 registros
Status API 3: 200 | página 4 | 50 registros
Registros API 3: 200


In [11]:
# GUARDAR DATASET API 3
API3_ARCHIVO = OUTPUT_DIR / 'dataset_api_3.csv'

df_api3 = pd.DataFrame(api3_registros)
df_api3.to_csv(API3_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API3_ARCHIVO)
print('Registros guardados:', len(df_api3))
df_api3.head()


Archivo generado: dataset_api_3.csv
Registros guardados: 200


,anilist_id,nombre,es_estudio_animacion,favoritos,obras_mas_populares,url
0,2,Kyoto Animation,True,23567,"Koe no Katachi, Violet Evergarden, Chuunibyou ...",https://anilist.co/studio/2
1,11,MADHOUSE,True,20087,"DEATH NOTE, HUNTER×HUNTER (2011), One Punch Man",https://anilist.co/studio/11
2,569,MAPPA,True,18675,"Jujutsu Kaisen, Shingeki no Kyojin: The Final ...",https://anilist.co/studio/569
3,43,ufotable,True,15187,"Kimetsu no Yaiba, Kimetsu no Yaiba, Kimetsu no...",https://anilist.co/studio/43
4,4,bones,True,14506,"Boku no Hero Academia, Hagane no Renkinjutsush...",https://anilist.co/studio/4


# Resumen final

La siguiente celda se ejecuta al final y muestra el resultado real de la carga.


In [12]:
total = len(api1_registros) + len(api2_registros) + len(api3_registros)

print('RESUMEN DE CARGA')
print('-' * 50)
print(f'API 1 (Shikimori): {len(api1_registros)} registros - {API1_ARCHIVO.name}')
print(f'API 2 (Kitsu):     {len(api2_registros)} registros - {API2_ARCHIVO.name}')
print(f'API 3 (AniList):   {len(api3_registros)} registros - {API3_ARCHIVO.name}')
print('-' * 50)
print(f'TOTAL: {total} registros')

if len(api1_registros) < 200:
    print('ADVERTENCIA: API 1 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api2_registros) < 200:
    print('ADVERTENCIA: API 2 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api3_registros) < 200:
    print('ADVERTENCIA: API 3 tiene menos de 200 registros. Documente la excepción si corresponde.')


RESUMEN DE CARGA
--------------------------------------------------
API 1 (Shikimori): 200 registros - dataset_api_1.csv
API 2 (Kitsu):     200 registros - dataset_api_2.csv
API 3 (AniList):   200 registros - dataset_api_3.csv
--------------------------------------------------
TOTAL: 600 registros


# Entrega en GitHub

El repositorio debe contener:

```text
MLY1101-Taller1-ApellidoNombre/
├── MLY1101_001V_T01_ApellidoNombre.ipynb
├── dataset_api_1.csv
├── dataset_api_2.csv
├── dataset_api_3.csv
└── README.md
```

Si el repositorio es privado, agregar como colaborador a **titiriemann**.


## Checklist final

- [x] Definí una pregunta u objetivo común.
- [x] Utilicé 3 APIs del repositorio indicado (Shikimori, Kitsu, AniList).
- [x] Obtuve al menos 200 registros por API.
- [x] Generé 3 archivos independientes.
- [x] No uní ni crucé los datasets.
- [ ] El notebook ejecuta desde cero (verificar con *Ejecutar todas*).
- [ ] Subí notebook, datasets y README.md a GitHub.
- [ ] Entregué la URL del repositorio.
